In [ ]:
import pandas as pd
import numpy as np 
import pyreadstat
from factor_analyzer.factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo
from factor_analyzer import (ConfirmatoryFactorAnalyzer, ModelSpecificationParser)

def loadings_to_frame(model:FactorAnalyzer, data_frame:pd.DataFrame, labels_dict:dict):
    """
    Takes the FA model, the original data frame and the labels dictionary and returns the loading matrix as a dataframe with the attributes as labels in the index. 
    """
    output = pd.DataFrame(model.loadings_).T
    output.columns = data_frame.columns
    output = output.T
    output.index = output.index.map(labels_dict)
    return output

def merge_factors_to_data(model:FactorAnalyzer, factor_input_df:pd.DataFrame, original_df:pd.DataFrame, sample_label:str, grid_label:str, rotation_label:str, no_of_factors:int):
    """
    Transforms the data into the factor variables and adds it to the original dataframe. Returns the updated original dataframe. 
    """
    output = pd.DataFrame(model.transform(factor_input_df.fillna(factor_input_df.mean())))
    output.index = factor_input_df.index
    output.columns = [f'{sample_label}_{grid_label}_{no_of_factors}F_{rotation_label}_{n}' for n in range(output.shape[1])]
    original_df = pd.merge(
        left=original_df,
        right=output,
        left_index=True,
        right_index=True,
        how='left'
    )

    return original_df


def factor_analysis_generation(data_frame:pd.DataFrame, sample_filters:dict, grids:dict, att_labels:dict ):

    loading_matrices = {}
    accumulated_variances = {}
    updated_data = data_frame.copy()
    grid_KMO_diagnostics = {}
    attributes_KMO_diagnostics = {}

    rotations = ['promax','varimax','equamax']
    number_of_factors = np.arange(2,100)

    # Factor analysis main loop 
    for sample in sample_filters.keys():
        for grid in grids.keys():
            for rotation in rotations:
                for no in number_of_factors:

                    # If the iteration number is greater or equal to the number of attributes it skips this iteraiton.
                    if no >= len(grids[grid]):
                        continue
                    
                    # Filters the sample using the filter mask defined in the sample filters dictionary and selects the attribute columns.
                    filtered_df = data_frame[sample_filters[sample]][grids[grid]]

                    fa = FactorAnalyzer(n_factors=no, rotation=rotation, method='principal', impute='mean')

                    try:
                        fa.fit(filtered_df)
                    except Exception as e:
                        accumulated_variances[f'{sample}_{grid}_{no}F_{rotation}'] = e # Logs the error if there was an issue while fitting the model 
                        continue

                    accumulated_variances[f'{sample}_{grid}_{no}F_{rotation}'] = fa.get_factor_variance()[2][-1] # Logs the accumulated variance for the model

                    if fa.get_factor_variance()[2][-1] >= 0.6:
                        loadings_matrix = loadings_to_frame(
                            model=fa,
                            data_frame=filtered_df,
                            labels_dict=att_labels
                        )

                        loading_matrices[f'{sample}_{grid}_{no}F_{rotation}'] = loadings_matrix

                        updated_data = merge_factors_to_data(
                            model=fa, 
                            factor_input_df=filtered_df, 
                            original_df=updated_data, 
                            sample_label=sample, 
                            grid_label=grid, 
                            rotation_label=rotation, 
                            no_of_factors=no
                            )
                        
    for sample in sample_filters.keys():
        for grid in grids.keys():
            filtered_df = data_frame[sample_filters[sample]][grids[grid]]

            # Calculate KMO scores
            attribute_scores, grid_score = calculate_kmo(filtered_df)

            grid_KMO_diagnostics[f'{sample}_{grid}_KMO Score'] =  grid_score

            attributes_KMO_diagnostics.update(dict(zip(filtered_df.columns, attribute_scores)))
            

    
    return loading_matrices, accumulated_variances, updated_data, grid_KMO_diagnostics, attributes_KMO_diagnostics

### IO Input

In [ ]:
data, meta = pyreadstat.read_sav('ZAD3SVX_FINAL_SPSS_FINAL.sav')

data['Pais'] = data['Pais'].map(meta.variable_value_labels['Pais'])
data['Variable_Type'] = data['Variable_Type'].map(meta.variable_value_labels['Variable_Type'])
data['Brand'] = data['Brand'].map(meta.variable_value_labels['Brand'])

# data.dropna(
#     subset=data.filter(like='BRAND_IMAGERY').columns,
#     how='all',
#     inplace = True
#     )

In [ ]:
# Diccionario de variables
grids = {
    'ACT':['BRAND_IMAGERY13' ,'BRAND_IMAGERY16' ,'BRAND_IMAGERY18' ,'BRAND_IMAGERY_201' ,'BRAND_IMAGERY_204' ,'BRAND_IMAGERY_220' ,'BRAND_IMAGERY_224'],
    'EMO':['BRAND_IMAGERY03' ,'BRAND_IMAGERY07' ,'BRAND_IMAGERY09' ,'BRAND_IMAGERY_202' ,'BRAND_IMAGERY_203' ,'BRAND_IMAGERY_215' ,'BRAND_IMAGERY_216' ,'BRAND_IMAGERY_221' ,'BRAND_IMAGERY_222' ,'BRAND_IMAGERY_223' ,'BRAND_IMAGERY_225'],
    'FUN':['BRAND_IMAGERY01' ,'BRAND_IMAGERY02' ,'BRAND_IMAGERY04' ,'BRAND_IMAGERY06' ,'BRAND_IMAGERY10' ,'BRAND_IMAGERY11' ,'BRAND_IMAGERY12' ,'BRAND_IMAGERY14' ,'BRAND_IMAGERY15' ,'BRAND_IMAGERY17' ,'BRAND_IMAGERY19' ,'BRAND_IMAGERY20' ,'BRAND_IMAGERY21' ,'BRAND_IMAGERY_218'],
    'OCC':['BRAND_IMAGERY05' ,'BRAND_IMAGERY_208' ,'BRAND_IMAGERY_209' ,'BRAND_IMAGERY_211' ,'BRAND_IMAGERY_212' ,'BRAND_IMAGERY_213' ,'BRAND_IMAGERY_214' ,'BRAND_IMAGERY_217','BRAND_IMAGERY_226'],
    'IMG':['BRAND_IMAGERY08' ,'BRAND_IMAGERY_205' ,'BRAND_IMAGERY_206' ,'BRAND_IMAGERY_207' ,'BRAND_IMAGERY_210' ,'BRAND_IMAGERY_219']
}

labels= {
    'BRAND_IMAGERY13': 'Attractive Packaging',
    'BRAND_IMAGERY16': 'Convenient Presentations',
    'BRAND_IMAGERY18': 'Ideal Size',
    'BRAND_IMAGERY_201': 'Fair Value',
    'BRAND_IMAGERY_204': 'Easy To Find',
    'BRAND_IMAGERY_220': 'It Has Offers Or Promotions',
    'BRAND_IMAGERY_224': 'Engaging Advertising',
    'BRAND_IMAGERY03': 'Irresistible',
    'BRAND_IMAGERY07': 'Stimulates Your Senses',
    'BRAND_IMAGERY09': 'Offers Something Different',
    'BRAND_IMAGERY_202': 'Reliable',
    'BRAND_IMAGERY_203': 'Fun',
    'BRAND_IMAGERY_215': 'It Is Popular',
    'BRAND_IMAGERY_216': 'It Is Sophisticated',
    'BRAND_IMAGERY_221': 'Great Career',
    'BRAND_IMAGERY_222': 'You Identify',
    'BRAND_IMAGERY_223': 'Remember Your Childhood',
    'BRAND_IMAGERY_225': 'Rebellious Irreverent Style',
    'BRAND_IMAGERY01': 'Quenches Thirst',
    'BRAND_IMAGERY02': 'Mixer',
    'BRAND_IMAGERY04': 'Light And Balanced',
    'BRAND_IMAGERY06': 'Refreshing',
    'BRAND_IMAGERY10': 'Tastes More Like Fruit',
    'BRAND_IMAGERY11': 'Recharges You With Energy',
    'BRAND_IMAGERY12': 'Flavor',
    'BRAND_IMAGERY14': 'Natural Ingredients',
    'BRAND_IMAGERY15': 'Sweetness Options',
    'BRAND_IMAGERY17': 'Natural Flavor',
    'BRAND_IMAGERY19': 'High Quality',
    'BRAND_IMAGERY20': 'Variety Of Flavors',
    'BRAND_IMAGERY21': 'Leaves A Pleasant Sensation',
    'BRAND_IMAGERY_218': 'Helps You Relax Your Body And Mind',
    'BRAND_IMAGERY05': 'To Accompany Meals',
    'BRAND_IMAGERY_208': 'To Share With Friends',
    'BRAND_IMAGERY_209': 'To Share With The Whole Family',
    'BRAND_IMAGERY_211': 'To Enhance Moments',
    'BRAND_IMAGERY_212': 'For Everyday',
    'BRAND_IMAGERY_213': 'Is For When I Go From One Place To Another',
    'BRAND_IMAGERY_214': 'Is For A Celebratory Occasion',
    'BRAND_IMAGERY_217': 'Related To Moments Of Entertainment',
    'BRAND_IMAGERY08': 'I Like To Be Seen Consuming It',
    'BRAND_IMAGERY_205': 'Innovative',
    'BRAND_IMAGERY_206': 'Modern',
    'BRAND_IMAGERY_207': 'For Someone Like Me',
    'BRAND_IMAGERY_210': 'When I Want To Treat Myself',
    'BRAND_IMAGERY_219': 'Helps You See The Good Side Of Life',
    'BRAND_IMAGERY_226': 'For enjoying when I am alone'
}

filter_masks = {
    'Brazil': (data['Variable_Type'] == 'Marca') & (data['Brand']=='H2OH!') & (data['MUESTRA']==1)
}

### Main loop

In [ ]:
loading_matrices, accumulated_variances, updated_data, grid_KMO_scores, attribute_KMO_scores = factor_analysis_generation(
    data_frame=data,
    sample_filters=filter_masks,
    grids=grids,
    att_labels=labels
)

### IO Output

In [ ]:
with pd.ExcelWriter('Loading_Matrices.xlsx') as writer:
    pd.Series(accumulated_variances).to_excel(writer, sheet_name='Explained Variance')
    pd.Series(grid_KMO_scores).to_excel(writer, sheet_name='Grid KMO Daignostics')
    pd.Series(attribute_KMO_scores).to_excel(writer, sheet_name='Variables KMO Diagnostics')
    for excercise in loading_matrices.keys():
        loading_matrices[excercise].to_excel(writer, sheet_name=excercise)

In [ ]:
updated_data.to_pickle('./Updated_Data_w_Factors.pkl')

### Analysis 

In [ ]:
updated_data = pd.read_pickle('./Updated_Data_w_Factors.pkl')

In [ ]:
selected_solutions = {
    'México': ['México_ACT_4F_promax', 'México_EMO_7F_promax', 'México_FUN_6F_promax', 'México_OCC_4F_promax', 'México_IMG_4F_promax'],
    'Argentina':['Argentina_ACT_4F_promax', 'Argentina_EMO_5F_promax', 'Argentina_EMO_6F_promax', 'Argentina_FUN_6F_promax', 'Argentina_OCC_5F_promax', 'Argentina_IMG_4F_promax'],
    'Colombia':['Colombia_ACT_3F_promax', 'Colombia_EMO_6F_promax', 'Colombia_FUN_6F_promax', 'Colombia_FUN_6F_equamax', 'Colombia_OCC_5F_promax', 'Colombia_IMG_3F_promax', 'Colombia_IMG_4F_promax'],
    'Guatemala':['Guatemala_ACT_4F_promax', 'Guatemala_EMO_6F_promax', 'Guatemala_FUN_8F_promax', 'Guatemala_OCC_5F_promax', 'Guatemala_IMG_4F_promax']
}
selected_solutions_dfs = {}

for country in selected_solutions.keys():
    for solution in selected_solutions[country]:
        temp_df = updated_data[updated_data['Pais']==country].groupby('Brand').mean(numeric_only=True).filter(like=solution)*100
        selected_solutions_dfs[f'{solution}'] = temp_df.T

In [ ]:
with pd.ExcelWriter('Factor Performance.xlsx') as writer:
    for df in selected_solutions_dfs.keys():
        selected_solutions_dfs[df].to_excel(writer, sheet_name=df)

### Confirmatory factor analysis

In [ ]:
model_dict = {
    'ACT':['BRAND_IMAGERY13' ,'BRAND_IMAGERY16' ,'BRAND_IMAGERY18' ,'BRAND_IMAGERY_201' ,'BRAND_IMAGERY_204' ,'BRAND_IMAGERY_220' ,'BRAND_IMAGERY_224'],
    'EMO':['BRAND_IMAGERY03' ,'BRAND_IMAGERY07' ,'BRAND_IMAGERY09' ,'BRAND_IMAGERY_202' ,'BRAND_IMAGERY_203' ,'BRAND_IMAGERY_215' ,'BRAND_IMAGERY_216' ,'BRAND_IMAGERY_221' ,'BRAND_IMAGERY_222' ,'BRAND_IMAGERY_223' ,'BRAND_IMAGERY_225'],
    'FUN':['BRAND_IMAGERY01' ,'BRAND_IMAGERY02' ,'BRAND_IMAGERY04' ,'BRAND_IMAGERY06' ,'BRAND_IMAGERY10' ,'BRAND_IMAGERY11' ,'BRAND_IMAGERY12' ,'BRAND_IMAGERY14' ,'BRAND_IMAGERY15' ,'BRAND_IMAGERY17' ,'BRAND_IMAGERY19' ,'BRAND_IMAGERY20' ,'BRAND_IMAGERY21' ,'BRAND_IMAGERY_218'],
    'OCC':['BRAND_IMAGERY05' ,'BRAND_IMAGERY_208' ,'BRAND_IMAGERY_209' ,'BRAND_IMAGERY_211' ,'BRAND_IMAGERY_212' ,'BRAND_IMAGERY_213' ,'BRAND_IMAGERY_214' ,'BRAND_IMAGERY_217','BRAND_IMAGERY_226'],
    'IMG':['BRAND_IMAGERY08' ,'BRAND_IMAGERY_205' ,'BRAND_IMAGERY_206' ,'BRAND_IMAGERY_207' ,'BRAND_IMAGERY_210' ,'BRAND_IMAGERY_219']
}

data_cfa = data[data['Pais']=='Brasil'].copy()
data_cfa = data_cfa.filter(like='BRAND_IMAGERY')

model_spec = ModelSpecificationParser.parse_model_specification_from_dict(data_cfa, model_dict)

In [ ]:
cfa = ConfirmatoryFactorAnalyzer(model_spec, impute='mean')

cfa.fit(data_cfa)

In [ ]:
var_list = []

for key in model_dict.keys():
    var_list.extend(model_dict[key])

loadings_to_frame(cfa,data_cfa[var_list],labels).to_clipboard()
loadings_to_frame(cfa,data_cfa[var_list],labels)